# Tutorial 4, Q1b: Icki-Toffee, weighted cost and popularity

Release after the practical.

**The problem.** The same constraints as the other Q1b notebooks, but the two aims are combined with a weight
$\alpha$ between 0 and 1 (lecture PDF page 20): maximise $(1-\alpha)\sum_i P_i x_i - \alpha \sum_i C_i x_i$
subject to $\sum_i W_i x_i \ge 320$ and $\sum_i W_i x_i \le 1.5 \times 320 = 480$ (weight, g; the upper limit is the Q1a weight cap), $\sum_i K_i x_i \le 2850$ (calories), $\sum_i F_i x_i \le 0.45 \sum_i W_i x_i$ (fat at most 45% of the weight), $x_i \ge 0$ and whole numbers. $\alpha$ is how much cost matters: $\alpha = 0$ ignores cost (maximise popularity),
$\alpha = 1$ ignores popularity (minimise cost).

| Index `i` | Candy | Index `i` | Candy |
| --- | --- | --- | --- |
| 0 | Buttery butterscotch | 6 | Strawberry surprise |
| 1 | Caramel softie | 7 | Raspberry mousse |
| 2 | Orange dream | 8 | Ginger toffee |
| 3 | Coconut Ice | 9 | Rose-water posy |
| 4 | Lemon sorbet | 10 | Walnut whirl |
| 5 | Lime sherbet | 11 | Grape bunch |

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** The notebook uses $\alpha = 0$, so cost is ignored and this is the maximum-popularity model
with the weight cap (page 16, *With Weight Cap*): popularity **305**, for example 30 Coconut Ice, 5 Rose-water
posy and 5 Grape bunch (475 g). The *Try it* at the end changes $\alpha$.

*Original script: `5 Source Files/Python/icki_toffee_weighted.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

*Note: this problem has more than one optimal selection, so the selection shown can differ between solvers and versions (GLPK, for example, gives x[3] = 29, x[5] = 2, x[9] = 4, x[11] = 5). Any selection that scores 305 and meets the constraints is correct.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

Twelve whole-number variables: `x[i]` is the number of candies of type `i` (table above).

In [ ]:
# Integer decision variables: x[i] = number of candies of type i, >= 0
model.x = Var(range(12), domain=NonNegativeIntegers)

The data from the tutorial sheet, one entry per candy in the order of the table above: cost $C_i$ and popularity $P_i$ (Table 2) for the objective; fat $F_i$ (g), calories $K_i$ and weight $W_i$ (g) (Table 1) for the constraints. A '-' in the tables is 0.

In [ ]:
# Data per candy: cost and popularity (Table 2, objective); fat, calories, weight (Table 1, constraints)
cost = [12, 12, 12, 15, 13, 13, 16, 16, 15, 10, 18, 20]
pop = [10, 10, 8, 9, 5, 6, 7, 8, 10, 3, 8, 4]
fat = [20, 16, 10, 7, 6, 5, 13, 15, 18, 0, 19, 0]
cals = [100, 100, 80, 50, 60, 40, 80, 90, 80, 145, 120, 120]
weight = [24, 22, 15, 12, 10, 10, 14, 17, 20, 8, 20, 15]

Choose $\alpha$, then build the weighted objective. Popularity is added and cost subtracted, so maximising pushes popularity up and cost down; $\alpha$ sets the balance. With `alpha = 0` the cost term is multiplied by 0.

In [ ]:
# choose alpha between 0 and 1: the weight on cost (0 = popularity only, 1 = cost only)
alpha = 0

# Objective: (1 - alpha) * total popularity - alpha * total cost
model.obj = Objective(
    expr=(1 - alpha) * sum(pop[i] * model.x[i] for i in model.x)
          - alpha * sum(cost[i] * model.x[i] for i in model.x),
    sense=maximize
)

The four constraints. `weight_max` is the Q1a weight cap. The last one is named `fat_weight_min`, but it is an **upper** limit: total fat at most 45% of total weight.

In [ ]:
# Define the inequality constraints
model.weight_min = Constraint(expr=sum(weight[i] * model.x[i] for i in model.x) >= 320)  # weight at least 320 g
model.weight_max = Constraint(expr=sum(weight[i] * model.x[i] for i in model.x) <= 1.5 * 320)  # Q1a cap: at most 480 g
model.cals_max = Constraint(expr=sum(cals[i] * model.x[i] for i in model.x) <= 2850)  # calories at most 2850
model.fat_weight_min = Constraint(expr=sum(fat[i] * model.x[i] for i in model.x) <= 0.45 * sum(weight[i] * model.x[i] for i in model.x))  # fat at most 45% of the weight (an upper limit)

Solve with HiGHS; `tee=True` prints the solver's log (branch and bound, because the variables are integers). Near the end the log should say the model status is *Optimal*.

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')  # HiGHS runs in the browser (the original script used GLPK)
solver.solve(model, tee=True)

`pprint()` prints the whole model: the variables, the objective and the constraints written out with the data filled in. Use it to check that the model is the one you meant.

In [ ]:
# Print the model
model.pprint()

`display()` prints the values: each `x[i]`, the objective, and the left-hand side of each constraint next to its limits. Pyomo moves everything in the fat constraint to one side, so it shows fat − 0.45 × weight ≤ 0: its value is how far below the limit the selection is (0 = binding).

In [ ]:
# Print the results
model.display()

## Try it

Change `alpha = 0` to `alpha = 0.6` (the slides' 40/60 split: 40% popularity, 60% cost) and choose *Run → Run All
Cells*. Which selection do you get? Then try `alpha = 0.5`. What do you notice?

<details><summary>Answer</summary>

With $\alpha = 0.6$ the model picks 9 Caramel softie, 10 Rose-water posy and 3 Grape bunch: the minimum-cost box
of Week 03 Q2b (cost 268, popularity 132, 323 g). $\alpha = 0.5$, "equal preference", gives the same box. Costs
(10 to 20) are bigger numbers than popularities (3 to 10), so the cost term wins unless $\alpha$ is small: any
$\alpha$ from about 0.42 gives the minimum-cost box. This is the issue on page 20 (*Cost and Popularity may be of
a different scale*); the normalised notebook fixes it.
</details>